In [ ]:
import os
import pandas as pd
import numpy as np

In [ ]:
#SET BASE DIRECTORY
#This notebook expects the GSE135779 data folders (adult_individual_h5ad, GSE135779_RAW, Results, etc.) to sit one level above this Notebooks folder. Update BASE_DIR below if your data lives elsewhere.

import os
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

In [ ]:
from pathlib import Path
import sys

sys.path.insert(0, str(Path(BASE_DIR) / "Notebooks"))
from publication_utils import configure_publication_notebook

FIGURE_DIR = configure_publication_notebook(BASE_DIR, "04D_GSE135779_cH_aSLE_COMPARISON")


In [ ]:
# Input folders
asle_dir = f"{BASE_DIR}/Results/aSLE_liana"
ch_dir   = f"{BASE_DIR}/Results/cH_liana"

# Output folder
out_dir = f"{BASE_DIR}/Results/cH_aSLE_comparison"
os.makedirs(out_dir, exist_ok=True)

# Input files
asle_file = os.path.join(asle_dir, "aSLE_liana_res.csv")
ch_file   = os.path.join(ch_dir, "cH_liana_res.csv")

# Load
res_asle = pd.read_csv(asle_file)
res_ch   = pd.read_csv(ch_file)

print("aSLE shape:", res_asle.shape)
print("cH shape:", res_ch.shape)
display(res_asle.head())
display(res_ch.head())

In [ ]:
def make_interaction_id(df):
    df = df.copy()
    df["interaction_id"] = (
        df["source"].astype(str) + "|" +
        df["target"].astype(str) + "|" +
        df["ligand_complex"].astype(str) + "|" +
        df["receptor_complex"].astype(str)
    )
    return df

res_asle = make_interaction_id(res_asle)
res_ch   = make_interaction_id(res_ch)

res_asle["condition"] = "aSLE"
res_ch["condition"] = "cH"

def summarize_interactions(df):
    """Average donor-level LIANA metrics for descriptive cohort summaries."""
    identifiers = ["interaction_id", "source", "target", "ligand_complex", "receptor_complex"]
    metrics = [c for c in ["lr_means", "cellphone_pvals", "expr_prod", "scaled_weight",
                            "lr_logfc", "spec_weight", "lrscore", "specificity_rank",
                            "magnitude_rank"] if c in df.columns]
    return df.groupby(identifiers, as_index=False, observed=True)[metrics].mean()
res_asle = summarize_interactions(res_asle)
res_ch = summarize_interactions(res_ch)
res_asle["condition"] = "aSLE"
res_ch["condition"] = "cH"


In [ ]:
keep_cols = [
    "interaction_id",
    "source", "target", "ligand_complex", "receptor_complex",
    "lr_means", "cellphone_pvals", "expr_prod", "scaled_weight",
    "lr_logfc", "spec_weight", "lrscore",
    "specificity_rank", "magnitude_rank",
    "condition"
]

res_asle_sub = res_asle[keep_cols].copy()
res_ch_sub   = res_ch[keep_cols].copy()

**Group comparison: cH (healthy children) vs. aSLE (adults with SLE).**

Compares the two *donor-averaged, within-condition* LIANA runs from notebooks 02A-02D by taking the difference in each interaction's `magnitude_rank` between conditions (a lower rank = a stronger consensus interaction in that condition). **This is a descriptive ranking, not a statistical hypothesis test** -- there is no p-value or FDR correction on the resulting "stronger in X" calls, since the donor-averaged rank difference itself is not the donor-level hypothesis test to test against. See notebooks 03B and 06A-06B for the donor-level statistical approach (Spearman + BH-FDR) that this group comparison should eventually be rebuilt on.

In [ ]:
# MERGED TABLE
merged = pd.merge(
    res_ch_sub.drop(columns=["condition"]),
    res_asle_sub.drop(columns=["condition"]),
    on="interaction_id",
    how="outer",
    suffixes=("_cH", "_aSLE")
)

merged["present_in_cH"] = merged["magnitude_rank_cH"].notna()
merged["present_in_aSLE"] = merged["magnitude_rank_aSLE"].notna()

# Positive/red = stronger in aSLE
# Negative/blue = stronger in cH
merged["rank_diff_cH_minus_aSLE"] = (
    merged["magnitude_rank_cH"] - merged["magnitude_rank_aSLE"]
)

merged.to_csv(os.path.join(out_dir, "cH_vs_aSLE_liana_merged.csv"), index=False)

print("Saved merged file")
print(merged.shape)
display(merged.head())

In [ ]:
# SHARED INTERACTIONS STRONGER IN aSLE

shared = merged[
    merged["present_in_cH"] & merged["present_in_aSLE"]
].copy()

shared["stronger_condition"] = np.where(
    shared["magnitude_rank_aSLE"] < shared["magnitude_rank_cH"],
    "aSLE",
    "cH"
)

shared["rank_difference"] = shared["magnitude_rank_cH"] - shared["magnitude_rank_aSLE"]
shared["abs_rank_difference"] = shared["rank_difference"].abs()

asle_stronger = shared[shared["stronger_condition"] == "aSLE"].copy()
asle_stronger = asle_stronger.sort_values(
    by=["rank_difference", "magnitude_rank_aSLE"],
    ascending=[False, True]
)

asle_stronger.to_csv(
    os.path.join(out_dir, "aSLE_stronger_shared_interactions.csv"),
    index=False
)

asle_stronger.head(50).to_csv(
    os.path.join(out_dir, "aSLE_stronger_top50.csv"),
    index=False
)

print("Saved aSLE stronger shared interactions")
display(asle_stronger.head(10))

In [ ]:
# SHARED INTERACTIONS STRONGER IN cH

ch_stronger = shared[shared["stronger_condition"] == "cH"].copy()
ch_stronger = ch_stronger.sort_values(
    by=["rank_difference", "magnitude_rank_cH"],
    ascending=[True, True]
)

ch_stronger.to_csv(
    os.path.join(out_dir, "cH_stronger_shared_interactions.csv"),
    index=False
)

ch_stronger.head(50).to_csv(
    os.path.join(out_dir, "cH_stronger_top50.csv"),
    index=False
)

print("Saved cH stronger shared interactions")
display(ch_stronger.head(10))

In [ ]:
# aSLE ONLY INTERACTIONS

asle_only = merged[
    (~merged["present_in_cH"]) & (merged["present_in_aSLE"])
].copy()

asle_only = asle_only.sort_values(by="magnitude_rank_aSLE", ascending=True)

asle_only.to_csv(
    os.path.join(out_dir, "aSLE_only_interactions.csv"),
    index=False
)

print("Saved aSLE-only interactions")
display(asle_only.head(10))

**Note on "only in condition X" counts.** These reflect LIANA's `expr_prop=0.1` expression-detection floor in one condition but not the other -- an interaction can fail to appear simply because a ligand or receptor gene didn't clear the 10%-of-cells expression threshold in that condition. This is a detection cutoff, not a significance test, so these counts should be read as "detected only in X" rather than "significantly present only in X."

In [ ]:
# cH ONLY INTERACTIONS

ch_only = merged[
    (merged["present_in_cH"]) & (~merged["present_in_aSLE"])
].copy()

ch_only = ch_only.sort_values(by="magnitude_rank_cH", ascending=True)

ch_only.to_csv(
    os.path.join(out_dir, "cH_only_interactions.csv"),
    index=False
)

print("Saved cH-only interactions")
display(ch_only.head(10))

In [ ]:
# SUMMARY TABLE

summary = pd.DataFrame({
    "metric": [
        "total_interactions_cH",
        "total_interactions_aSLE",
        "shared_interactions",
        "cH_only_interactions",
        "aSLE_only_interactions",
        "shared_stronger_in_cH",
        "shared_stronger_in_aSLE"
    ],
    "value": [
        len(res_ch_sub),
        len(res_asle_sub),
        len(shared),
        len(ch_only),
        len(asle_only),
        len(ch_stronger),
        len(asle_stronger)
    ]
})

summary.to_csv(
    os.path.join(out_dir, "cH_aSLE_comparison_summary.csv"),
    index=False
)

print("Saved summary table")
display(summary)

***VISUALIZATION***

In [ ]:
# BARPLOT

import matplotlib.pyplot as plt
import seaborn as sns

top_asle = asle_stronger.head(20).copy()

top_asle["label"] = (
    top_asle["source_aSLE"] + " → " +
    top_asle["target_aSLE"] + " | " +
    top_asle["ligand_complex_aSLE"] + "-" +
    top_asle["receptor_complex_aSLE"]
)

plt.figure(figsize=(10, 8))

sns.barplot(
    data=top_asle,
    y="label",
    x="rank_difference"
)

plt.title("Top aSLE-Enriched Cell-Cell Interactions")
plt.xlabel("Rank Difference (cH - aSLE)")
plt.ylabel("Interaction")

plt.tight_layout()
from publication_utils import save_publication_figure
save_publication_figure(FIGURE_DIR / "figure_01.png")
plt.show()

In [ ]:
# HEATMAP: aSLE vs cH
# Positive/red = stronger in aSLE
# Negative/blue = stronger in cH

heatmap_data = shared.copy()

heatmap_data["rank_difference"] = (
    heatmap_data["magnitude_rank_cH"] -
    heatmap_data["magnitude_rank_aSLE"]
)

heatmap_matrix = (
    heatmap_data
    .groupby(["source_aSLE", "target_aSLE"])["rank_difference"]
    .mean()
    .unstack()
    .fillna(0)
)

plt.figure(figsize=(11, 8))

sns.heatmap(
    heatmap_matrix,
    cmap="coolwarm",
    center=0,
    linewidths=0.4,
    linecolor="gray",
    cbar_kws={
        "label": "Mean rank difference\n(+ aSLE stronger, - cH stronger)"
    }
)

plt.title("Differential Cell–Cell Communication: aSLE vs cH")
plt.xlabel("Target Cell Type")
plt.ylabel("Source Cell Type")

plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)

plt.tight_layout()
from publication_utils import save_publication_figure
save_publication_figure(FIGURE_DIR / "figure_02.png")
plt.show()

In [ ]:
# NETWORK PLOT

import networkx as nx

net_df = asle_stronger.head(30)

G = nx.DiGraph()

for _, row in net_df.iterrows():
    G.add_edge(
        row["source_aSLE"],
        row["target_aSLE"],
        weight=row["rank_difference"]
    )

plt.figure(figsize=(10, 10))

pos = nx.spring_layout(G, seed=42)

edges = G.edges(data=True)
weights = [d["weight"] for (_, _, d) in edges]

nx.draw(
    G,
    pos,
    with_labels=True,
    node_size=2000,
    font_size=10,
    width=[w * 2 for w in weights]
)

plt.title("Top aSLE Cell–Cell Communication Network")
from publication_utils import save_publication_figure
save_publication_figure(FIGURE_DIR / "figure_03.png")
plt.show()

In [ ]:
# BOTH DIRECTIONS IN ONE PLOT

top_ch = ch_stronger.head(20).copy()
top_ch["direction"] = "cH stronger"

top_asle = asle_stronger.head(20).copy()
top_asle["direction"] = "aSLE stronger"

combined = pd.concat([top_asle, top_ch])

combined["label"] = (
    combined["source_aSLE"].fillna(combined["source_cH"]) + " → " +
    combined["target_aSLE"].fillna(combined["target_cH"])
)

plt.figure(figsize=(10, 8))

sns.barplot(
    data=combined,
    y="label",
    x="abs_rank_difference",
    hue="direction"
)

plt.title("Differential Cell Communication (aSLE vs cH)")
plt.xlabel("Absolute Rank Difference")

plt.tight_layout()
from publication_utils import save_publication_figure
save_publication_figure(FIGURE_DIR / "figure_04.png")
plt.show()

In [ ]:
import os
import pandas as pd

RESULTS_DIR = f"{BASE_DIR}/Results"
fpath = os.path.join(RESULTS_DIR, 'cH_aSLE_comparison', 'cH_vs_aSLE_liana_merged.csv')
df = pd.read_csv(fpath)

# Check the situation first
print("Total rows:", len(df))
print("Shared:", (df['present_in_cH'] & df['present_in_aSLE']).sum())
print("cH only:", (df['present_in_cH'] & ~df['present_in_aSLE']).sum())
print("aSLE only:", (~df['present_in_cH'] & df['present_in_aSLE']).sum())

# Check for naming mismatches
print("\ncH source cell types:")
print(sorted(df['source_cH'].dropna().unique()))
print("\naSLE source cell types:")
print(sorted(df['source_aSLE'].dropna().unique()))

# Filter to shared only, rank by mean lrscore
df_shared = df[df['present_in_cH'] & df['present_in_aSLE']].copy()
df_shared['_ranking_score'] = df_shared[['lrscore_cH', 'lrscore_aSLE']].mean(axis=1)
top30 = df_shared.sort_values('_ranking_score', ascending=False).head(30).drop(columns=['_ranking_score'])

out_path = os.path.join(RESULTS_DIR, 'cH_aSLE_comparison', 'top30_cH_vs_aSLE_shared.csv')
top30.to_csv(out_path, index=False)
print(f"\nSaved {len(top30)} rows")
print(top30[['interaction_id', 'lrscore_cH', 'lrscore_aSLE', 'rank_diff_cH_minus_aSLE']].head(10).to_string())

## Statistical rebuild: donor-level Mann-Whitney test + FDR correction

Everything above this point is the original **descriptive** rank-difference comparison -- kept for continuity, but it is not a hypothesis test (see the note at the top of this notebook). This section replaces it with a real cross-group test: donor-level pseudobulk interaction scores (from 5B) for each condition, compared with an unpaired **Mann-Whitney U test** per interaction (patients in the two conditions are different people, not matched pairs), with **Benjamini-Hochberg FDR correction** across all interactions tested. This mirrors the approach already used correctly in 06A/06B.

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
from statistical_utils import pairwise_effect_sizes, rank_biserial_from_u

from analysis_config import MIN_PATIENTS_PER_GROUP
MIN_PATIENTS = MIN_PATIENTS_PER_GROUP

scores_a = pd.read_csv(f"{BASE_DIR}/Results/severity_analysis/cH_per_patient_scores.csv")
scores_b = pd.read_csv(f"{BASE_DIR}/Results/severity_analysis/aSLE_per_patient_scores.csv")

shared_ids = set(scores_a["interaction_id"]) & set(scores_b["interaction_id"])
print(f"Interactions with per-patient scores in both cH and aSLE: {len(shared_ids)}")

records = []
for iid, group_a in scores_a[scores_a["interaction_id"].isin(shared_ids)].groupby("interaction_id"):
    group_b = scores_b[scores_b["interaction_id"] == iid]

    n_a = group_a["sample"].nunique()
    n_b = group_b["sample"].nunique()
    if n_a < MIN_PATIENTS or n_b < MIN_PATIENTS:
        continue
    if group_a["score"].std() == 0 and group_b["score"].std() == 0:
        continue

    try:
        stat, pval = mannwhitneyu(group_a["score"], group_b["score"], alternative="two-sided")
    except ValueError:
        continue

    effects = pairwise_effect_sizes(
        group_a["score"], group_b["score"], iid
    )
    row = group_a.iloc[0]
    records.append({
        "interaction_id": iid,
        "source": row["source"], "target": row["target"],
        "ligand_complex": row["ligand_complex"], "receptor_complex": row["receptor_complex"],
        f"n_cH": n_a, f"n_aSLE": n_b,
        f"median_cH": group_a["score"].median(), f"median_aSLE": group_b["score"].median(),
        "stronger_condition": "cH" if group_a["score"].median() > group_b["score"].median() else "aSLE",
        "mannwhitney_stat": stat,
        "rank_biserial_a_minus_b": rank_biserial_from_u(stat, n_a, n_b),
        **effects,
        "p_value": pval,
    })

stats_result = pd.DataFrame(records)
if len(stats_result) > 0:
    stats_result["fdr_q_value"] = multipletests(stats_result["p_value"], method="fdr_bh")[1]
    stats_result = stats_result.sort_values("fdr_q_value")

stats_out_path = f"{BASE_DIR}/Results/cH_aSLE_comparison/cH_vs_aSLE_mannwhitney_stats.csv"
stats_result.to_csv(stats_out_path, index=False)

print(f"Tested {len(stats_result)} interactions with >= {MIN_PATIENTS} patients per group")
print(f"Significant at FDR < 0.05: {(stats_result['fdr_q_value'] < 0.05).sum() if len(stats_result) else 0}")
print(f"Significant at FDR < 0.10: {(stats_result['fdr_q_value'] < 0.10).sum() if len(stats_result) else 0}")
print(f"Saved: {stats_out_path}")
stats_result.head(20)